In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    if files and any(f.endswith(('.jpg', '.png', '.jpeg')) for f in files):
        print(f"{root} → {len(files)} files")

/kaggle/input/datasets/utkarshsaxenadn/landscape-recognition-image-dataset-12k-images/Landscape Classification/Landscape Classification/Testing Data/Mountain → 100 files
/kaggle/input/datasets/utkarshsaxenadn/landscape-recognition-image-dataset-12k-images/Landscape Classification/Landscape Classification/Testing Data/Coast → 100 files
/kaggle/input/datasets/utkarshsaxenadn/landscape-recognition-image-dataset-12k-images/Landscape Classification/Landscape Classification/Testing Data/Desert → 100 files
/kaggle/input/datasets/utkarshsaxenadn/landscape-recognition-image-dataset-12k-images/Landscape Classification/Landscape Classification/Testing Data/Forest → 100 files
/kaggle/input/datasets/utkarshsaxenadn/landscape-recognition-image-dataset-12k-images/Landscape Classification/Landscape Classification/Testing Data/Glacier → 100 files
/kaggle/input/datasets/utkarshsaxenadn/landscape-recognition-image-dataset-12k-images/Landscape Classification/Landscape Classification/Training Data/Mountain

In [2]:
# ============================================================
# Steganography Detection — Based on pbrucla/steganography-cnns
# Uses EfficientNetV2 | 5-class: Clean, LSB, PVD, DCT, FFT
# ============================================================

# ── 1. Imports ───────────────────────────────────────────────
import os, glob, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import f1_score
from torch.optim.lr_scheduler import StepLR

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

# ── 2. Dataset paths (matched to attached Kaggle datasets) ───
# Class labels: 0=Clean, 1=LSB, 2=PVD, 3=DCT, 4=FFT

LANDSCAPE = '/kaggle/input/datasets/utkarshsaxenadn/landscape-recognition-image-dataset-12k-images/Landscape Classification/Landscape Classification'
PVD_BASE  = '/kaggle/input/datasets/petrdufek/stego-pvd-dataset/Stego-pvd-dataset'
STEGO_BASE = '/kaggle/input/datasets/marcozuppelli/stegoimagesdataset'
LSB_BASE  = '/kaggle/input/datasets/diegozanchett/digital-steganography'

def get_files(folder, ext=('.jpg','.jpeg','.png'), limit=None):
    files = [os.path.join(folder, f) for f in os.listdir(folder)
             if f.lower().endswith(ext)]
    if limit:
        files = files[:limit]
    return files

# Build dataset: list of (filepath, label)
def build_dataset(split='train', limit=3000):
    data = []

    if split == 'train':
        # Clean images — landscape training data
        for folder in ['Mountain','Coast','Desert','Forest','Glacier']:
            path = os.path.join(LANDSCAPE, 'Training Data', folder)
            if os.path.exists(path):
                for f in get_files(path, limit=limit//5):
                    data.append((f, 0))  # label 0 = Clean

        # LSB stego
        for folder in ['lsb', 'lsb_grayscale']:
            path = os.path.join(LSB_BASE, folder)
            if os.path.exists(path):
                for f in get_files(path, limit=limit//2):
                    data.append((f, 1))  # label 1 = LSB

        # PVD stego
        path = os.path.join(PVD_BASE, 'train', 'stegoTrain')
        for f in get_files(path, limit=limit):
            data.append((f, 2))  # label 2 = PVD

        # DCT stego (stego from marcozuppelli = DCT based)
        path = os.path.join(STEGO_BASE, 'train', 'train', 'stego')
        for f in get_files(path, limit=limit):
            data.append((f, 3))  # label 3 = DCT

        # SSB (FFT-like) stego
        path = os.path.join(LSB_BASE, 'ssbn')
        for f in get_files(path, limit=limit):
            data.append((f, 4))  # label 4 = FFT/SSB

    elif split == 'val':
        # Clean
        for folder in ['Mountain','Coast','Desert','Forest','Glacier']:
            path = os.path.join(LANDSCAPE, 'Validation Data', folder)
            if os.path.exists(path):
                for f in get_files(path, limit=100):
                    data.append((f, 0))

        # LSB
        path = os.path.join(LSB_BASE, 'ssb4')
        for f in get_files(path, limit=500):
            data.append((f, 1))

        # PVD
        path = os.path.join(PVD_BASE, 'val', 'stegoVal')
        for f in get_files(path, limit=500):
            data.append((f, 2))

        # DCT
        path = os.path.join(STEGO_BASE, 'val', 'val', 'stego')
        for f in get_files(path, limit=500):
            data.append((f, 3))

        # FFT/SSB — use clean val as proxy
        path = os.path.join(PVD_BASE, 'val', 'cleanVal')
        for f in get_files(path, limit=500):
            data.append((f, 4))

    random.shuffle(data)
    return data

print("Building datasets...")
train_data = build_dataset('train', limit=3000)
val_data   = build_dataset('val')

# Count per class
from collections import Counter
train_counts = Counter([l for _, l in train_data])
val_counts   = Counter([l for _, l in val_data])
print(f"Train: {len(train_data)} images  {dict(train_counts)}")
print(f"Val:   {len(val_data)} images  {dict(val_counts)}")

# ── 3. Dataset class (from repo) ─────────────────────────────
class resize_images(object):
    def __init__(self, target_size=(128, 128)):
        self.target_size = target_size

    def __call__(self, img):
        padding = (
            max(0, (self.target_size[0] - img.height) // 2),
            max(0, (self.target_size[1] - img.width) // 2),
            max(0, (self.target_size[0] - img.height + 1) // 2),
            max(0, (self.target_size[1] - img.width + 1) // 2),
        )
        transform = v2.Compose([
            v2.Pad(padding=padding, fill=0, padding_mode='constant'),
            v2.CenterCrop(self.target_size),
        ])
        return transform(img)

class StegoDataset(Dataset):
    def __init__(self, data, image_size=256):
        self.data = data
        self.transform = v2.Compose([
            resize_images((image_size, image_size)),
            v2.ToImage(),
            v2.ToDtype(torch.float32),
            v2.Normalize([0.485*255, 0.456*255, 0.406*255],
                         [0.229*255, 0.224*255, 0.225*255]),
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        filepath, label = self.data[idx]
        try:
            image = Image.open(filepath).convert('RGB')
            image = self.transform(image)
        except Exception:
            image = torch.zeros(3, 256, 256)
        return image, torch.tensor(label, dtype=torch.long)

train_dataset = StegoDataset(train_data)
val_dataset   = StegoDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}")

# ── 4. Model (EfficientNetV2 — same as repo) ─────────────────
NUM_CLASSES = 5
CLASS_LABELS = ['Clean', 'LSB', 'PVD', 'DCT', 'FFT']

model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)

# Replace classifier head — same approach as repo's get_model
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, NUM_CLASSES)
)

# Freeze backbone first (repo uses freeze_model then unroll per epoch)
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False

model = model.to(DEVICE)
print(f"Model: EfficientNetV2-S  |  Classes: {NUM_CLASSES}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ── 5. Loss with class weights (from repo) ───────────────────
dataset_sizes = [train_counts[i] for i in range(NUM_CLASSES)]
total = sum(dataset_sizes)
class_weights = torch.tensor([total / s for s in dataset_sizes]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Two param groups — repo uses different LR for head vs backbone
optimizer = torch.optim.Adam([
    {'params': model.classifier.parameters(), 'lr': 1e-3},
], lr=1e-3)

scheduler = StepLR(optimizer, step_size=2, gamma=0.5)

# ── 6. Train (from repo's train_one_epoch) ───────────────────
def accuracy_metric(predicted, labels):
    correct = (predicted == labels).sum().item()
    return correct, labels.size(0)

def train_one_epoch(epoch, model, loader):
    model.train()
    correct = total = 0
    running_loss = 0.0
    all_preds, all_labels = [], []

    with tqdm(loader, desc=f"Epoch {epoch+1} [Train]") as pbar:
        for imgs, labels in pbar:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs).squeeze()
            preds = torch.argmax(outputs, dim=1)
            new_correct, new_total = accuracy_metric(preds, labels)
            correct += new_correct
            total   += new_total
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            running_loss += loss.item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100*correct/total:.2f}%'})

    f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
    acc = 100 * correct / total
    print(f"  Train loss: {running_loss/len(loader):.4f}  Acc: {acc:.2f}%")
    for i, (label, score) in enumerate(zip(CLASS_LABELS, f1)):
        print(f"    {label} F1: {score:.3f}")
    return acc

def val_one_epoch(epoch, model, loader):
    model.eval()
    correct = total = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        with tqdm(loader, desc=f"Epoch {epoch+1} [Val]") as pbar:
            for imgs, labels in pbar:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs).squeeze()
                preds = torch.argmax(outputs, dim=1)
                new_correct, new_total = accuracy_metric(preds, labels)
                correct += new_correct
                total   += new_total
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                pbar.set_postfix({'acc': f'{100*correct/total:.2f}%'})

    f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
    acc = 100 * correct / total
    print(f"  Val Acc: {acc:.2f}%")
    for i, (label, score) in enumerate(zip(CLASS_LABELS, f1)):
        print(f"    {label} F1: {score:.3f}")
    return acc

# ── 7. Training loop (repo: freeze → train head → unroll backbone) ──
EPOCHS = 10
best_acc = 0

print(f"\n🚀 Training EfficientNetV2 — {EPOCHS} epochs")
print("=" * 50)

for epoch in range(EPOCHS):
    # After epoch 2, unfreeze backbone (repo's unroll strategy)
    if epoch == 2:
        print("\n🔓 Unfreezing backbone for fine-tuning...")
        for param in model.parameters():
            param.requires_grad = True
        optimizer = torch.optim.Adam([
            {'params': model.features.parameters(), 'lr': 1e-5},
            {'params': model.classifier.parameters(), 'lr': 1e-4},
        ])
        scheduler = StepLR(optimizer, step_size=2, gamma=0.5)
        print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    train_acc = train_one_epoch(epoch, model, train_loader)
    val_acc   = val_one_epoch(epoch, model, val_loader)
    scheduler.step()

    if val_acc > best_acc:
        best_acc = val_acc
        model.cpu()
        torch.save(model.state_dict(), '/kaggle/working/stego_efficientv2.pth')
        model.to(DEVICE)
        print(f"  💾 Best saved! Val Acc: {best_acc:.2f}%")

    print("-" * 50)

print(f"\n🎉 Done! Best Val Accuracy: {best_acc:.2f}%")
print("Download: Output tab → /kaggle/working → stego_efficientv2.pth")

Device: cuda
Building datasets...
Train: 15000 images  {4: 3000, 0: 3000, 1: 3000, 3: 3000, 2: 3000}
Val:   2500 images  {2: 500, 1: 500, 0: 500, 3: 500, 4: 500}
Train batches: 469  Val batches: 79
Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 192MB/s]


Model: EfficientNetV2-S  |  Classes: 5
Trainable params: 6,405

🚀 Training EfficientNetV2 — 10 epochs


Epoch 1 [Train]: 100%|██████████| 469/469 [02:53<00:00,  2.71it/s, loss=0.8250, acc=64.10%]


  Train loss: 0.6832  Acc: 64.10%
    Clean F1: 0.928
    LSB F1: 0.601
    PVD F1: 0.492
    DCT F1: 0.488
    FFT F1: 0.682


...

  Val Acc: 41.68%
    Clean F1: 1.000
    LSB F1: 0.051
    PVD F1: 0.469
    DCT F1: 0.345
    FFT F1: 0.000
--------------------------------------------------


Epoch 10 [Train]: 100%|██████████| 469/469 [03:41<00:00,  2.11it/s, loss=0.3547, acc=77.77%]


  Train loss: 0.3886  Acc: 77.77%
    Clean F1: 0.998
    LSB F1: 0.853
    PVD F1: 0.601
    DCT F1: 0.562
    FFT F1: 0.871


Epoch 10 [Val]: 100%|██████████| 79/79 [00:14<00:00,  5.36it/s, acc=42.00%]

  Val Acc: 42.00%
    Clean F1: 1.000
    LSB F1: 0.062
    PVD F1: 0.467
    DCT F1: 0.364
    FFT F1: 0.000
--------------------------------------------------

🎉 Done! Best Val Accuracy: 43.56%
Download: Output tab → /kaggle/working → stego_efficientv2.pth


In [3]:
# Fix: rebuild val with correct class assignments
def build_val_fixed():
    data = []
    
    # Clean — landscape val
    for folder in ['Mountain','Coast','Desert','Forest','Glacier']:
        path = os.path.join(LANDSCAPE, 'Validation Data', folder)
        if os.path.exists(path):
            for f in get_files(path, limit=200):
                data.append((f, 0))

    # LSB
    path = os.path.join(LSB_BASE, 'lsb')
    for f in get_files(path, limit=500):
        data.append((f, 1))

    # PVD
    path = os.path.join(PVD_BASE, 'val', 'stegoVal')
    for f in get_files(path, limit=500):
        data.append((f, 2))

    # DCT — use stego from marcozuppelli val
    path = os.path.join(STEGO_BASE, 'val', 'val', 'stego')
    for f in get_files(path, limit=500):
        data.append((f, 3))

    # FFT — use ssbn
    path = os.path.join(LSB_BASE, 'ssbn')
    for f in get_files(path, limit=500):
        data.append((f, 4))

    random.shuffle(data)
    return data

val_data_fixed = build_val_fixed()
val_dataset_fixed = StegoDataset(val_data_fixed)
val_loader_fixed = DataLoader(val_dataset_fixed, batch_size=32, shuffle=False, num_workers=2)

from collections import Counter
print("Fixed val distribution:", dict(Counter([l for _,l in val_data_fixed])))

# Evaluate current best model
model_eval = efficientnet_v2_s(weights=None)
in_features = model_eval.classifier[1].in_features
model_eval.classifier = nn.Sequential(nn.Dropout(p=0.2), nn.Linear(in_features, 5))
model_eval.load_state_dict(torch.load('/kaggle/working/stego_efficientv2.pth', map_location='cpu'))
model_eval = model_eval.to(DEVICE)
model_eval.eval()

correct = total = 0
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader_fixed:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = torch.argmax(model_eval(imgs), dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
print(f"\nFixed Val Accuracy: {100*correct/total:.2f}%")
for label, score in zip(CLASS_LABELS, f1):
    print(f"  {label} F1: {score:.3f}")

Fixed val distribution: {1: 500, 0: 1000, 4: 500, 2: 500, 3: 500}

Fixed Val Accuracy: 78.40%
  Clean F1: 0.985
  LSB F1: 0.866
  PVD F1: 0.557
  DCT F1: 0.435
  FFT F1: 0.862
